In [ ]:
import warnings
warnings.filterwarnings('ignore')

# %matplotlib inline
from pathlib import Path
from time import sleep

import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm as tqdm_bar
import time as _time
import random
import tempfile
from scipy.optimize import curve_fit

from orenge.general import *


# Experiment controls -------------------------------------------------------------

import qcodes as qc
from qcodes.dataset import (
    Measurement,
    initialise_or_create_database_at,
    load_or_create_experiment,
)
from qcodes.logger import start_all_logging
start_all_logging()
import qcodes.instrument_drivers.stanford_research as stanford_research

from spinapi import *

import pyvisa # for the SR830 lockin amplifier and Anritsu RF signal generator
pyvisa_rm = pyvisa.ResourceManager()

import os
import sys
ultolib_dir = os.path.dirname('C:/Users/nv-group02/Desktop/NV-Lab/ultolib')
sys.path.insert(0, ultolib_dir)
from ultolib import (anritsu, korad, spincore)
from ultolib.spincore import pulse

#### original

In [ ]:
pulse_blaster = spincore.PulseBlasterESRPRO(name='pulse_blaster', board_number=0)
pulse_blaster.core_clock(500)   # 500 MHz clock -> 2 ns resolution

lock_in_amp = stanford_research.SR830(
    name='lock_in_amp',
    address='ASRL5::INSTR',
    terminator='\r',
)

- this was the original working code before qcodes update 0.57.0, following `Instrument_Introduction_Lab_1.ipynb`
- qcodes api handles driver construction with serial configs
- qcodes then added an update to the sr830 driver, the `GeneratedSetPoints` buffer feature
- new object `GeneratedSetPoints` added to `SR830.__init__`, so `SRAT ?` query is called before serial port settings configured
- since sample rate query triggers before config, instrument doesn't respond and causes visa timeout

then ran a diagnostic to test for hardware issues

```python
_lia_test = pyvisa_rm.open_resource('ASRL5::INSTR')
_lia_test.baud_rate        = 9600
_lia_test.data_bits        = 8
_lia_test.read_termination  = '\r'
_lia_test.write_termination = '\r\n'
print(_lia_test.query('*IDN?'))
_lia_test.close()
```

this returned the traceback:

```raw
pyvisa.errors.VisaIOError: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.

During handling of the above exception, another exception occurred:
...
  File "h5py/h5.pyx", line 1, in init h5py.h5
ImportError: DLL load failed while importing defs: The specified procedure could not be found.
```



`pyvisa.errors.VisaIOError` 
- the sr830 didn't respond to `*IDN?` over raw pyvista
- instrument's `OUTX` register was likely not set to RS-232 mode
- instead was configured to respond on GPIB and ignored serial queries

`ImportError`
- failed to display full traceback of above error
- `h5py` DLL was broken because environment was corrupted

#### try 1

try to apply config settings manually to fix port opening with the wrong defaults

In [ ]:
lock_in_amp = stanford_research.SR830(
    name='lock_in_amp',
    address='ASRL5::INSTR',
    terminator='\r',
)
lock_in_amp.visa_handle.baud_rate         = 9600
lock_in_amp.visa_handle.data_bits         = 8
lock_in_amp.visa_handle.read_termination  = '\r'
lock_in_amp.visa_handle.write_termination = '\r\n'

still got traceback `pyvisa.errors.VisaIOError` in instantiation, so manual configs with `visa_handle` aren't seen yet

#### try 2

inject serial settings before port is handed to driver and sr830 is instantiated

In [ ]:
_pre = pyvisa_rm.open_resource('ASRL5::INSTR')
_pre.baud_rate = 9600
_pre.data_bits = 8
_pre.close()

lock_in_amp = stanford_research.SR830(
    name='lock_in_amp',
    address='ASRL5::INSTR',
    terminator='\r'
)

closing `_pre` discards settings, so try a patch to modify default behavior

In [ ]:
pulse_blaster = spincore.PulseBlasterESRPRO(name='pulse_blaster', board_number=0)
pulse_blaster.core_clock(500)

import pyvisa as _pyvisa
_orig_open = _pyvisa.ResourceManager.open_resource

def _patched_open(self, resource_name, **kwargs):
    if resource_name == 'ASRL5::INSTR':
        kwargs.setdefault('baud_rate',         9600)
        kwargs.setdefault('data_bits',         8)
        kwargs.setdefault('read_termination',  '\r')
        kwargs.setdefault('write_termination', '\r\n')
    return _orig_open(self, resource_name, **kwargs)

_pyvisa.ResourceManager.open_resource = _patched_open

lock_in_amp = stanford_research.SR830(
    name='lock_in_amp',
    address='ASRL5::INSTR',
    terminator='\r',
)

_pyvisa.ResourceManager.open_resource = _orig_open   # restore immediately

traceback:

```raw
pyvisa.errors.VisaIOError: ('VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.', "asking 'SRAT ?' to <SR830: lock_in_amp>", 'getting lock_in_amp_buffer_SR')
```

serial configs fixed, but instrument isn't responding still, so this means `OUTX` register wasn't properly set to RS-232

#### try 3

the new query added to the qcodes update triggers with `update_units_if_constant_sample_rate()`, so try to suppress this code by searching for occurrences of `self.update_units_if_constant_sample_rate()` and replace with `pass`

In [ ]:
import inspect, importlib

_mod = importlib.import_module('qcodes.instrument_drivers.stanford_research.SR830')
_path = inspect.getfile(_mod)
print(_path)

with open(_path, 'r') as f:
    src = f.read()

print(f'Occurrences remaining: {src.count("self.update_units_if_constant_sample_rate()")}')

src_patched = src.replace(
    'self.update_units_if_constant_sample_rate()',
    'pass  # patched: suppressed premature SRAT? query'
)

with open(_path, 'w') as f:
    f.write(src_patched)

```raw
c:\Users\nv-group02\anaconda3\envs\mqst\Lib\site-packages\qcodes\instrument_drivers\stanford_research\SR830.py 
Occurrences remaining: 0
```

No occurrences but error still fires so delete cache

In [ ]:
import os, inspect, importlib

_mod = importlib.import_module('qcodes.instrument_drivers.stanford_research.SR830')
_path = inspect.getfile(_mod)
_cache_dir = os.path.join(os.path.dirname(_path), '__pycache__')

for f in os.listdir(_cache_dir):
    if 'SR830' in f:
        full = os.path.join(_cache_dir, f)
        os.remove(full)
        print(f'Deleted: {full}')

```raw
Unexpected exception formatting exception. Falling back to standard exception
Traceback (most recent call last):
...
File "c:\Users\nv-group02\anaconda3\envs\mqst\Lib\site-packages\qcodes\dataset\database_extract_runs.py", line 11, in <module>
    from opentelemetry import trace
ImportError: cannot import name 'trace' from 'opentelemetry' (unknown location)
During handling of the above exception, another exception occurred:
Traceback (most recent call last):
...
 File "h5py/h5.pyx", line 1, in init h5py.h5
ImportError: DLL load failed while importing defs: The specified procedure could not be found.
```

suppression patch corrupted environment, had to recreate with `.yaml` file

#### solution

since environment is refreshed/new, going back to pre-config attempt instead of modifying qcodes. explicitly set the register to `OUTX 0` before qcodes constructs

---

Note: `address=None` is required because `SR830.__init__` declares `address` as a required positional argument
```python
lock_in_amp = stanford_research.SR830(
    name='lock_in_amp',
    terminator='\r',
    resource=_resource,
)
```
gives traceback:
```raw
TypeError: SR830.__init__() missing 1 required positional argument: 'address'
```

---

Note: `VisaInstrument` raises a `TypeError` if both `address` and `resource` are non-None. Passing `address=None` satisfies SR830's signature while letting the `resource=` path take over in the base class.
```python
lock_in_amp = stanford_research.SR830(
    name='lock_in_amp',
    address='ASRL5::INSTR',
    terminator='\r',
    resource=_resource,
)
```
gives traceback:
```raw
TypeError: 'address' and 'resource' are mutually exclusive
```

In [ ]:
_resource = pyvisa_rm.open_resource('ASRL5::INSTR')
_resource.baud_rate = 9600
_resource.data_bits = 8
_resource.write('OUTX 0\r')

lock_in_amp = stanford_research.SR830(
    name='lock_in_amp',
    address=None,
    terminator='\r',
    resource=_resource,
)

yay! this works!